In [1]:
NUM_CLUSTS = 6  # 6 to 10
SMOOTH_WIDTH = 5  # Odd>1: 3,5,7,9,...
above_dir = "../input/hms-harmful-brain-activity-classification/"
GLOBAL_SEED = 42



In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

import pyarrow
import pyarrow.parquet as pq
import pyarrow.dataset as pads

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression

plt.ioff()



In [3]:
HBA_number = 6
HBA_names = ["seizure", "lpd", "gpd", "lrda", "grda", "other"]
HBA_expert_names = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
iHBA_of_expert = {"Seizure": 0, "LPD": 1, "GPD": 2, "LRDA": 3, "GRDA": 4, "Other": 5}
HBA_votes = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
HBA_probs = [
    "seizure_prob",
    "lpd_prob",
    "gpd_prob",
    "lrda_prob",
    "grda_prob",
    "other_prob",
]
the4chains = ["LL", "RL", "LP", "RP"]

np.set_printoptions(precision=6, suppress=True)
np.random.seed(GLOBAL_SEED)




In [4]:
def kld_score(solution, submission, eps=1e-8):
    """
    Calculate the average KL divergence score.
    Assumes solution/submission have identical probability column names.
    """
    sol = solution.to_numpy(dtype=np.float64)
    sub = submission.to_numpy(dtype=np.float64)

    sol = np.clip(sol, eps, 1.0)
    sol = sol / sol.sum(axis=1, keepdims=True)

    sub = np.clip(sub, eps, 1.0)
    sub = sub / sub.sum(axis=1, keepdims=True)

    return float(np.mean(np.sum(sol * (np.log(sol) - np.log(sub)), axis=1)))




In [5]:
def read_hms_meta():
    """
    Read in the train.csv and test.csv files.
    Add total_vote, _prob columns, and vote entropy to train_meta.
    Add extra cols to test to allow the same processing as train:
        eeg[spectro]_sub_id, eeg[spectro]_label_offset_seconds, label_id
    """
    test_meta = pd.read_csv(above_dir + "test.csv")
    test_meta_len = len(test_meta)
    print("Test has length", test_meta_len)
    test_meta["eeg_sub_id"] = 0
    test_meta["eeg_label_offset_seconds"] = 0.0
    test_meta["spectrogram_sub_id"] = 0
    test_meta["spectrogram_label_offset_seconds"] = 0.0
    test_meta["label_id"] = test_meta.eeg_id

    train_meta = pd.read_csv(above_dir + "train.csv")
    train_meta_len = len(train_meta)
    print("Train has length", train_meta_len, " with:")

    train_meta["total_vote"] = (
        train_meta["seizure_vote"]
        + train_meta["lpd_vote"]
        + train_meta["gpd_vote"]
        + train_meta["lrda_vote"]
        + train_meta["grda_vote"]
        + train_meta["other_vote"]
    )
    train_meta["max_vote"] = np.max(
        np.array(
            [
                train_meta["seizure_vote"],
                train_meta["lpd_vote"],
                train_meta["gpd_vote"],
                train_meta["lrda_vote"],
                train_meta["grda_vote"],
                train_meta["other_vote"],
            ]
        ),
        axis=0,
    )

    for this_col in [
        "label_id",
        "eeg_id",
        "spectrogram_id",
        "patient_id",
        "total_vote",
    ]:
        print(
            "   ", len(train_meta[this_col].unique()), "unique " + this_col + " values."
        )

    for col_pre in HBA_names:
        train_meta[col_pre + "_prob"] = (
            train_meta[col_pre + "_vote"] / train_meta["total_vote"]
        )

    print("Calculating voting entropy values ...")

    probs = train_meta[HBA_probs].to_numpy(dtype=np.float64)
    probs = np.clip(probs, 1.0e-8, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)
    train_meta["entropy"] = np.nansum(probs * (-np.log(probs)), axis=1)

    return train_meta, test_meta




In [6]:
def prob_prob_scatter(name1, name2, probs2plot, clust_ids, iclust_order=[0]):
    """
    Make a prob1 vs prob2 scatter plot (diagnostic only).
    """
    hba_clrs = [
        "orange",
        "blue",
        "red",
        "black",
        "green",
        "purple",
        "green",
        "red",
        "blue",
        "orange",
    ]
    if len(iclust_order) > 2:
        kmclrs = hba_clrs.copy()
        for iord, iclust in enumerate(iclust_order):
            kmclrs[iclust] = hba_clrs[iord]

    clstclrs = [kmclrs[ilab] for ilab in clust_ids]

    ixax = iHBA_of_expert[name1]
    iyax = iHBA_of_expert[name2]
    lenprob = len(probs2plot)
    plt.figure(figsize=(5, 5))
    plt.scatter(
        np.sqrt(probs2plot[HBA_probs[ixax]]) + 0.04 * (np.random.rand(lenprob) - 0.5),
        np.sqrt(probs2plot[HBA_probs[iyax]]) + 0.04 * (np.random.rand(lenprob) - 0.5),
        s=3,
        c=clstclrs,
        alpha=0.02,
    )
    for iclust in range(0, len(clust_centers)):
        plt.plot(
            np.sqrt([clust_centers[iclust, ixax]]),
            np.sqrt([clust_centers[iclust, iyax]]),
            c=kmclrs[iclust],
            marker="x",
            markersize=15,
        )
    plt.xlabel("sqrt( " + name1 + " )")
    plt.ylabel("sqrt( " + name2 + " )")
    plt.show()
    return kmclrs




In [7]:
train_meta, test_meta = read_hms_meta()



Test has length 9850
Train has length 96950  with:
    96950 unique label_id values.
    15396 unique eeg_id values.
    10024 unique spectrogram_id values.
    1910 unique patient_id values.
    25 unique total_vote values.
Calculating voting entropy values ...


In [8]:
if False:
    plt.figure(figsize=(5, 2))
    plt.hist(train_meta["spectrogram_sub_id"], bins=55, log=True)
    plt.title("Histogram of spectrogram_sub_id")
    plt.show()

    plt.figure(figsize=(5, 2))
    plt.hist(train_meta["eeg_sub_id"], bins=55, log=True)
    plt.title("Histogram of eeg_sub_id")
    plt.show()



In [9]:
num_clusts = NUM_CLUSTS  # 6 to 10

clust_rows_bool = (
    ((train_meta.eeg_sub_id < 56) & (train_meta.eeg_sub_id % 7 == 6))
    | (train_meta.eeg_sub_id == 0)
) & (train_meta.eeg_id % 4 < 2)



In [10]:
prob_vectors = train_meta.loc[clust_rows_bool, HBA_probs]

print("\nUsing {} HBA samples for clustering.".format(len(prob_vectors)))
print(
    "These include {} unique eeg_ids".format(
        train_meta.loc[clust_rows_bool, "eeg_id"].nunique()
    ),
    "and {} unique patient ids.".format(
        train_meta.loc[clust_rows_bool, "patient_id"].nunique()
    ),
)

prob_array = prob_vectors.to_numpy(dtype=np.float64)

kmeans = KMeans(
    n_clusters=num_clusts,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=GLOBAL_SEED,
)
kmeans.fit(prob_array)

clust_centers = kmeans.cluster_centers_
for iclust in range(num_clusts):
    clust_centers[iclust, :] = clust_centers[iclust, :] / np.sum(
        clust_centers[iclust, :]
    )

iclust_of_order = []
for icol in range(HBA_number):
    iclust_of_order.append(np.argmax(clust_centers[:, icol]))
clust_by_max = np.argsort(-1 * np.max(clust_centers, axis=1))
for iord in range(HBA_number, num_clusts):
    iclust_of_order.append(clust_by_max[iord])

kmnames = HBA_expert_names.copy()
for ihyb in range(1, (num_clusts - HBA_number) + 1):
    kmnames.append("Hybrid-" + str(ihyb))




Using 11462 HBA samples for clustering.
These include 7705 unique eeg_ids and 1569 unique patient ids.


In [11]:
train_meta["clust_id"] = kmeans.predict(
    train_meta[HBA_probs].to_numpy(dtype=np.float64)
)

all_probs = train_meta[HBA_probs]
all_ids = train_meta["clust_id"]

if False:
    kmclrs = prob_prob_scatter("Seizure", "GPD", all_probs, all_ids, iclust_of_order)
    kmclrs = prob_prob_scatter("LPD", "GRDA", all_probs, all_ids, iclust_of_order)
    kmclrs = prob_prob_scatter("LRDA", "Other", all_probs, all_ids, iclust_of_order)
else:
    kmclrs = [
        "orange",
        "blue",
        "red",
        "black",
        "green",
        "purple",
        "green",
        "red",
        "blue",
        "orange",
    ]



In [12]:
solution_train = train_meta[["eeg_id"] + HBA_votes].copy()
for col_pre in HBA_names:
    solution_train.loc[:, col_pre + "_vote"] = train_meta[col_pre + "_prob"]

submission_train = solution_train.copy()
clust_ids = train_meta["clust_id"].to_numpy()

for iprob in range(HBA_number):
    this_col_probs = clust_centers[:, iprob]
    submission_train[HBA_votes[iprob]] = this_col_probs[clust_ids]

print(
    "Score if HBA samples are correctly assigned cluster prob.s:",
    np.round(kld_score(solution_train[HBA_votes], submission_train[HBA_votes]), 4),
)



Score if HBA samples are correctly assigned cluster prob.s: 0.306


/tmp/ipykernel_11/3392045363.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1. 1. 1. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  solution_train.loc[:, col_pre + "_vote"] = train_meta[col_pre + "_prob"]
/tmp/ipykernel_11/3392045363.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  solution_train.loc[:, col_pre + "_vote"] = train_meta[col_pre + "_prob"]
/tmp/ipykernel_11/3392045363.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  solution_train.loc[:, col_pre + "_vote"] = train_meta

In [13]:
smooth_width = SMOOTH_WIDTH  # Here smooth_width is just for looking,
spectro_meta = train_meta[clust_rows_bool].copy()
every_nth = 37



In [14]:
freqs = np.array(range(100)) * 0.19525 + 0.59
spect_trend = 150.0 / (1.0**2.3 + freqs ** (2.3))



In [15]:
if False:
    plt.figure(figsize=(10, 8))
    all_medians = []
    all_means = []
    all_clusts = []
    print(
        "Plotting {} x 4 processed spectra".format(int(len(spectro_meta) / every_nth))
    )
    for ilocrow in range(0, len(spectro_meta), every_nth):
        this_row = spectro_meta.iloc[ilocrow]
        this_clust = this_row.clust_id
        spectro_id_str = str(this_row.spectrogram_id)
        spectro_file = above_dir + "train_spectrograms/" + spectro_id_str + ".parquet"
        pads_spectro = pads.dataset(spectro_file)
        this_spectro = pads_spectro.to_table().to_pandas()
        loc_offset = int(this_row.spectrogram_label_offset_seconds / 2)
        for ispec in range(4):
            ibeg = ([1, 101, 201, 301])[ispec]
            iend = ibeg + 100
            middle4s = (
                this_spectro.iloc[loc_offset + 149, ibeg:iend]
                + this_spectro.iloc[loc_offset + 150, ibeg:iend]
            ) / (2.0 * spect_trend)
            middle4s = np.clip(middle4s, 0.001, 1000.0)
            middle4s = middle4s.replace([np.nan, -np.inf, np.inf], 0.001)
            spect_mean = np.log10(np.mean(middle4s))
            all_means.append(spect_mean)
            spect_median = np.log10(np.median(middle4s))
            all_medians.append(spect_median)
            all_clusts.append(this_clust)
            middle4s = np.log10(middle4s)
            middle4spre = middle4s - spect_mean
            middle4s = middle4spre.rolling(
                smooth_width, min_periods=smooth_width, center=True, closed=None
            ).mean()
            for ibin in range(int((smooth_width - 1) / 2)):
                middle4s[ibin] = middle4spre[ibin]
            plt.plot((freqs), middle4s, c=kmclrs[this_clust], lw=3, alpha=0.01)
        if (ilocrow / every_nth + 1) % 100 == 0:
            print("... {} done...".format(int(ilocrow / every_nth) + 1))
    plt.plot([0.0, 20.0], [0.0, 0.0], c="black", lw=3, alpha=0.2)
    downsel_freqs = np.insert(
        freqs[int((smooth_width - 1) / 2) : 100 : smooth_width], 0, freqs[0]
    )
    plt.plot(downsel_freqs, len(downsel_freqs) * [0.0], ".k")
    plt.ylim(-1.0, 1.0)
    plt.xlim(0.0, 20.5)
    plt.title(
        "Middle-4s Spectra (color-coded by the {} clusters, smooth={})".format(
            num_clusts, smooth_width
        )
    )
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("log10[] Amplitude (/ref-spectrum, /mean, and smoothed n=3) ]")
    plt.savefig("middle4s_spectra.png")
    plt.show()



In [16]:
if False:
    print("\nMedian of the Means(below): {:.4f}".format(np.median(all_means)))
    plt.figure(figsize=(6, 2))
    plt.hist(np.clip(all_means, -2.0, 2.0), bins=100)
    plt.xlim(-2.05, 2.05)
    plt.xlabel("Mean")
    plt.title("Histogram of the Means of the log(Spectra/Ref-spect)")
    plt.show()

    print("\nMedian of the Medians(below): {:.4f}".format(np.median(all_medians)))
    plt.figure(figsize=(6, 2))
    plt.hist(np.clip(all_medians, -2.0, 2.0), bins=100)
    plt.xlim(-2.05, 2.05)
    plt.xlabel("log10( Median )")
    plt.title("Histogram of the Medians of the log(Spectra/Ref-spect)")
    plt.show()

    mmclrs = [kmclrs[ilab] for ilab in all_clusts]
    plt.figure(figsize=(3, 3))
    plt.scatter(all_medians, all_means, s=2, c=mmclrs, alpha=0.3)
    plt.xlim(-1.0, 1)
    plt.ylim(-1.0, 1)
    plt.xlabel("Median")
    plt.ylabel("Mean")
    plt.show()



In [17]:
_SPECTRO_SLICE_CACHE = {}
_SPECTRO_SLICE_CACHE_ORDER = []
_SPECTRO_SLICE_CACHE_MAX = 5000  # bounded to keep memory under control


def _read_spectro_two_rows_400cols(spectro_file, r1, r2):
    """
    Return a (2, 400) numpy array for absolute rows [r1,r2] and columns [1..400]
    from the spectrogram parquet. Uses a bounded cache.

    Change (execution stability): if the parquet is missing/corrupt, return zeros.
    This keeps core feature logic identical when files are valid, but prevents crashes
    and downstream NaNs that can hurt KL or prevent a submission.
    """
    r1 = int(r1)
    r2 = int(r2)
    if r2 < r1:
        r1, r2 = r2, r1

    key = (spectro_file, r1, r2)
    if key in _SPECTRO_SLICE_CACHE:
        return _SPECTRO_SLICE_CACHE[key]

    try:
        pf = pq.ParquetFile(spectro_file)
    except Exception:
        out = np.zeros((2, 400), dtype=np.float64)
        _SPECTRO_SLICE_CACHE[key] = out
        _SPECTRO_SLICE_CACHE_ORDER.append(key)
        return out

    total_rows = pf.metadata.num_rows
    if total_rows <= 0:
        out = np.zeros((2, 400), dtype=np.float64)
        return out

    r1c = max(0, min(r1, total_rows - 1))
    r2c = max(0, min(r2, total_rows - 1))

    rg_sizes = [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)]
    cum = np.cumsum([0] + rg_sizes)  # cum[i] start row of row_group i

    def _rg_for_abs_row(r_abs):
        return int(np.searchsorted(cum, r_abs, side="right") - 1)

    rg1 = _rg_for_abs_row(r1c)
    rg2 = _rg_for_abs_row(r2c)

    all_schema_names = list(pf.schema.names)

    preferred_cols = [str(i) for i in range(1, 401)]
    use_preferred = all(c in all_schema_names for c in preferred_cols)

    def _select_400_cols(df):
        cols = list(df.columns)
        drop_candidates = {"time", "Time", "timestamp", "Timestamp"}
        cols2 = [c for c in cols if c not in drop_candidates]

        def _to_int_or_none(x):
            try:
                return int(x)
            except Exception:
                return None

        numeric_pairs = []
        for c in cols2:
            v = _to_int_or_none(c)
            if v is not None:
                numeric_pairs.append((v, c))
        if len(numeric_pairs) >= 400:
            numeric_pairs.sort(key=lambda t: t[0])
            sel = [c for _, c in numeric_pairs[:400]]
        else:
            sel = cols2[:400]
        if len(sel) < 400:
            sel = sel + [sel[-1]] * (400 - len(sel))
        return sel

    def _read_abs_row(pf_, rg, r_abs):
        if use_preferred:
            tab = pf_.read_row_group(rg, columns=preferred_cols)
            df = tab.to_pandas()
            local = int(r_abs - cum[rg])
            local = max(0, min(local, len(df) - 1))
            return df.iloc[local, :].to_numpy(dtype=np.float64, copy=False)

        tab = pf_.read_row_group(rg)  # all columns
        df = tab.to_pandas()
        local = int(r_abs - cum[rg])
        local = max(0, min(local, len(df) - 1))
        sel_cols = _select_400_cols(df)
        return df.loc[df.index[local], sel_cols].to_numpy(dtype=np.float64, copy=False)

    try:
        row1 = _read_abs_row(pf, rg1, r1c)
        row2 = _read_abs_row(pf, rg2, r2c)
        out = np.vstack([row1, row2]).astype(np.float64, copy=False)
    except Exception:
        out = np.zeros((2, 400), dtype=np.float64)

    if out.shape != (2, 400):
        out = np.zeros((2, 400), dtype=np.float64)

    _SPECTRO_SLICE_CACHE[key] = out
    _SPECTRO_SLICE_CACHE_ORDER.append(key)
    if len(_SPECTRO_SLICE_CACHE_ORDER) > _SPECTRO_SLICE_CACHE_MAX:
        old = _SPECTRO_SLICE_CACHE_ORDER.pop(0)
        _SPECTRO_SLICE_CACHE.pop(old, None)

    return out




In [18]:
def assemble_features(meta_frame, traintest="train", smooth_width=5, SHOW_PLOT=True):
    """
    Create a dataframe of spectrogram features from the meta_frame rows.
    Will include clust_id (i.e, the y) if it is in the input meta_frame.

    Change (score improvement): also carry HBA_probs into the returned feature frame
    when present in meta_frame. This fixes an X/y misalignment bug caused by resetting
    indices separately for meta vs features, which can otherwise train the RF regressor
    on mismatched targets and severely worsen KL.
    """
    freqs = np.array(range(100)) * 0.19525 + 0.59
    spect_trend = 150.0 / (1.0**2.3 + freqs ** (2.3))
    freqs[0] = 0.0
    freqs4 = np.array(4 * list(freqs))
    spect_trend4 = np.array(4 * list(spect_trend))

    baseinds = np.insert(
        np.arange(int((smooth_width - 1) / 2), 100, smooth_width), 0, 0
    )
    select_inds = np.concatenate(
        (baseinds, 100 + baseinds, 200 + baseinds, 300 + baseinds)
    )
    freqs4ds = freqs4[select_inds]

    if SHOW_PLOT:
        plt.figure(figsize=(10, 8))

    rows = []

    for n_done, irow in enumerate(meta_frame.index, start=1):
        this_row = meta_frame.loc[irow]
        spectro_id_str = str(int(this_row.spectrogram_id))
        spectro_file = (
            above_dir + traintest + "_spectrograms/" + spectro_id_str + ".parquet"
        )

        loc_offset = int(this_row.spectrogram_label_offset_seconds / 2)
        r1 = loc_offset + 149
        r2 = loc_offset + 150

        arr2x400 = _read_spectro_two_rows_400cols(spectro_file, r1, r2)
        middle4s = (arr2x400[0, :] + arr2x400[1, :]) / (2.0 * spect_trend4)

        middle4s = np.clip(middle4s, 0.001, 1000.0)
        middle4s = np.nan_to_num(middle4s, nan=0.001, posinf=0.001, neginf=0.001)

        spect_mean = float(np.mean(middle4s))
        spect_median = float(np.median(middle4s))

        the4means = np.empty(4, dtype=np.float64)
        the4medians = np.empty(4, dtype=np.float64)
        for ispec in range(4):
            ibeg = 100 * ispec
            iend = ibeg + 100
            seg = middle4s[ibeg:iend]
            the4means[ispec] = float(np.mean(seg))
            the4medians[ispec] = float(np.median(seg))

        middle4spre = np.log10(middle4s / spect_mean)

        middle4spre_s = pd.Series(middle4spre)
        middle4s_sm = middle4spre_s.rolling(
            smooth_width, min_periods=smooth_width, center=True, closed=None
        ).mean()
        for ioff in range(0, 400, 100):
            for ibin in range(int((smooth_width - 1) / 2)):
                middle4s_sm.iloc[ibin + ioff] = middle4spre_s.iloc[ibin + ioff]

        middle4sds = middle4s_sm.iloc[select_inds].to_numpy(dtype=np.float64)

        row_dict = {str(i): float(v) for i, v in enumerate(middle4sds)}
        row_dict["Mean"] = float(np.log10(spect_mean))
        row_dict["Median"] = float(np.log10(spect_median))
        the4means = np.log10(the4means)
        the4medians = np.log10(the4medians)
        for ispec in range(4):
            row_dict[the4chains[ispec] + "mean"] = float(the4means[ispec])
            row_dict[the4chains[ispec] + "median"] = float(the4medians[ispec])

        if "eeg_id" in meta_frame.columns:
            row_dict["eeg_id"] = int(this_row.eeg_id)

        if "clust_id" in meta_frame.columns:
            row_dict["clust_id"] = int(this_row.clust_id)

        for c in HBA_probs:
            if c in meta_frame.columns:
                row_dict[c] = float(this_row[c])

        rows.append(row_dict)

        if n_done % 200 == 0:
            print(f"... {n_done} done...")

        if SHOW_PLOT:
            this_clr = "blue"
            title_start = "Middle-4s Feature Values"
            if "clust_id" in meta_frame.columns:
                this_clust = int(this_row.clust_id)
                this_clr = kmclrs[this_clust]
                plt.title(
                    title_start
                    + " (color-coded by the "
                    + "{} clusters, smooth={})".format(num_clusts, smooth_width)
                )
            else:
                plt.title(title_start + " (smooth={})".format(smooth_width))

            the_alpha = np.clip(0.05 * 350 / len(meta_frame), 0.003, 0.5)
            plt.plot(
                freqs4ds + smooth_width * 0.15 * (np.random.rand(len(freqs4ds)) - 0.5),
                middle4sds,
                ".",
                c=this_clr,
                markersize=10,
                alpha=the_alpha,
            )
            plt.plot(
                -1.2 + smooth_width * 0.15 * (np.random.rand(4) - 0.5),
                the4means,
                ".",
                c=this_clr,
                markersize=10,
                alpha=the_alpha,
            )
            plt.ylim(-1.5, 1.5)
            plt.xlabel(
                "<-- Means are band < 0" + 20 * " " + "Frequency Bands (Hz)" + 50 * " "
            )
            plt.ylabel(
                "Amplitude (/ref-spectrum, /mean, and smoothed n={})".format(
                    smooth_width
                )
            )

    feats_frame = pd.DataFrame(rows)
    feats_frame.columns = feats_frame.columns.astype(str)

    if SHOW_PLOT:
        plt.savefig("middle4s_" + traintest + "_features.png")
        plt.show()

    return feats_frame




In [19]:
Xy_train_meta = (train_meta[clust_rows_bool])[::23].copy()
Xy_train_meta = Xy_train_meta.reset_index().drop(columns=["index"])
print("Number of samples used for training =", len(Xy_train_meta))

Xy_train_feats = assemble_features(
    Xy_train_meta, traintest="train", smooth_width=SMOOTH_WIDTH, SHOW_PLOT=False
)
Xy_train_meta.to_csv("Xy_train_meta.csv", header=True, index=False, float_format="%.6f")
Xy_train_feats.to_csv(
    "Xv_train_feats.csv", header=True, index=False, float_format="%.6f"
)



Number of samples used for training = 499
... 200 done...
... 400 done...


In [20]:
Xy_train_feats



,0,1,2,3,4,5,6,7,8,9,...,RPmean,RPmedian,eeg_id,clust_id,seizure_prob,lpd_prob,gpd_prob,lrda_prob,grda_prob,other_prob
0,-0.749755,-0.157718,0.186256,0.071226,-0.204631,-0.243480,-0.190642,-0.239125,-0.111397,-0.445216,...,0.390441,0.207988,722738444,1,0.000000,0.062500,0.000000,0.875,0.000000,0.062500
1,-0.035595,0.095505,0.260148,0.168489,0.136170,-0.077767,-0.129565,-0.065736,-0.114911,-0.302484,...,-0.622218,-0.674805,1347137760,0,0.000000,0.000000,0.000000,0.000,0.333333,0.666667
2,-0.816787,-0.532145,-0.143297,-0.184102,-0.219910,-0.260173,-0.450569,-0.610397,-0.769795,-0.608820,...,-0.236855,-0.370642,1076057868,1,0.000000,0.000000,0.000000,0.600,0.066667,0.333333
3,-1.708418,-1.169356,-0.673210,-0.370047,-0.402782,-0.014177,0.093893,0.109247,0.058366,-0.033225,...,0.039627,0.016987,4233815620,2,0.230769,0.000000,0.615385,0.000,0.000000,0.153846
4,-0.751516,-0.340376,0.163830,0.128949,0.064562,0.268711,0.002741,0.061578,-0.123694,-0.255723,...,-0.048698,-0.098421,3088095149,2,0.250000,0.000000,0.500000,0.000,0.000000,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,-0.296660,-0.153311,-0.313747,-0.225057,-0.071407,0.072462,0.279141,-0.025672,0.045220,-0.136676,...,-0.609198,-0.733878,1001369401,0,0.000000,0.333333,0.000000,0.000,0.000000,0.666667
495,-2.647144,-2.343564,-1.683720,-1.519719,-1.285685,-1.110440,-1.020107,-1.102360,-0.987115,-1.006458,...,1.338812,0.580539,2353888921,5,1.000000,0.000000,0.000000,0.000,0.000000,0.000000
496,-0.645249,-0.072504,0.315024,0.194124,0.156462,-0.035369,-0.186634,-0.285867,-0.118346,-0.133728,...,0.364345,0.260157,167136080,3,0.000000,0.000000,0.000000,0.000,1.000000,0.000000
497,-1.359609,-0.941599,-0.208777,0.308246,0.358818,0.446103,0.513499,0.405964,0.174371,0.133059,...,0.140659,0.019716,675867705,2,0.000000,0.000000,0.833333,0.000,0.000000,0.166667


In [21]:
valid_rows_bool = (
    ((train_meta.eeg_sub_id < 21) & (train_meta.eeg_sub_id % 7 == 4))
    | (train_meta.eeg_sub_id == 0)
) & (train_meta.eeg_id % 4 > 1)

Xy_valid_meta = (train_meta[valid_rows_bool])[::47].copy()
Xy_valid_meta = Xy_valid_meta.reset_index().drop(columns=["index"])
print("Number of samples used for Validation =", len(Xy_valid_meta))

Xy_valid_feats = assemble_features(
    Xy_valid_meta, traintest="train", smooth_width=SMOOTH_WIDTH, SHOW_PLOT=False
)
Xy_valid_meta.to_csv("Xy_valid_meta.csv", header=True, index=False, float_format="%.6f")
Xy_valid_feats.to_csv(
    "Xv_valid_feats.csv", header=True, index=False, float_format="%.6f"
)



Number of samples used for Validation = 250
... 200 done...


In [22]:
Xy_valid_feats



,0,1,2,3,4,5,6,7,8,9,...,RPmean,RPmedian,eeg_id,clust_id,seizure_prob,lpd_prob,gpd_prob,lrda_prob,grda_prob,other_prob
0,-1.459422,-0.970628,-0.111235,-0.256149,-0.259306,0.216685,-0.054303,0.011072,-0.144226,-0.140977,...,0.126222,0.080692,1628180742,5,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
1,-0.809985,-0.629347,-0.406703,-0.381245,-0.129025,0.112117,0.529424,0.468404,0.160491,0.182773,...,-0.113992,-0.103060,1041252567,4,0.0,1.000000,0.000000,0.000000,0.000000,0.000000
2,0.343953,0.573015,0.453977,0.279591,-0.114959,-0.315242,-0.202357,-0.268428,-0.398151,-0.370714,...,-0.632898,-0.703452,3987097859,1,0.0,0.000000,0.000000,0.333333,0.333333,0.333333
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-3.000000,-3.000000,2536169515,4,0.0,1.000000,0.000000,0.000000,0.000000,0.000000
4,-0.140787,0.048290,-0.219884,-0.215891,0.009938,0.476737,0.485879,0.599316,0.313554,-0.089858,...,-0.502869,-0.705074,1798323479,0,0.0,0.000000,0.000000,0.000000,0.066667,0.933333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,1.028412,1.147193,1.137064,0.741184,0.641852,0.822461,0.476813,0.420529,0.175971,0.563476,...,-0.929840,-0.951139,131163139,0,0.0,0.000000,0.090909,0.000000,0.000000,0.909091
246,-0.097857,0.154760,0.102436,0.115262,-0.113360,-0.146796,-0.092508,-0.432359,-0.284204,-0.347385,...,-0.691282,-0.724754,1749469215,5,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
247,-1.122584,-0.749586,-0.812537,-0.489503,-0.246979,-0.222219,-0.038777,0.188570,0.330554,0.252398,...,-0.130007,-0.182806,3606726294,4,0.0,0.928571,0.071429,0.000000,0.000000,0.000000
248,-1.183670,-1.010158,-0.468989,-0.544533,-0.304582,0.036726,-0.332324,0.037940,-0.199085,-0.086968,...,0.354553,0.305607,1570756647,5,1.0,0.000000,0.000000,0.000000,0.000000,0.000000


In [23]:
X = Xy_train_feats.drop(
    columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore"
).copy()
X.columns = X.columns.astype(str)
y = Xy_train_feats["clust_id"].astype(int)

BASE_FEATURE_COLS = list(X.columns)

# Change (execution stability): multi_class="auto" avoids failures when the subsample
# doesn't contain all clusters; this doesn't change the core LR approach.
lrmodel = LogisticRegression(
    C=1.0, max_iter=300, multi_class="auto", n_jobs=-1, random_state=GLOBAL_SEED
).fit(X.reindex(columns=BASE_FEATURE_COLS), y)



In [24]:
lrblur = 0.0

X_train_base = Xy_train_feats.drop(
    columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore"
).copy()
X_train_base.columns = X_train_base.columns.astype(str)
X_train_base = X_train_base.reindex(columns=BASE_FEATURE_COLS)
lrprobas_train = lrmodel.predict_proba(X_train_base)

Xy_train_wLRfeats = Xy_train_feats.copy()

# Change (score correctness): ensure LRp columns align to actual lrmodel.classes_,
# and fill missing clusters with 0.0 so downstream RF always sees consistent schema.
lr_classes = lrmodel.classes_.astype(int).tolist()
lr_proba_map_train = {c: lrprobas_train[:, j] for j, c in enumerate(lr_classes)}
for iadd in range(NUM_CLUSTS):
    base = lr_proba_map_train.get(iadd, np.zeros(len(X_train_base), dtype=np.float64))
    Xy_train_wLRfeats["LRp" + str(iadd)] = base + lrblur * (
        np.random.rand(len(base)) - 0.5
    )

X_valid_base = Xy_valid_feats.drop(
    columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore"
).copy()
X_valid_base.columns = X_valid_base.columns.astype(str)
X_valid_base = X_valid_base.reindex(columns=BASE_FEATURE_COLS)
lrprobas_valid = lrmodel.predict_proba(X_valid_base)

Xy_valid_wLRfeats = Xy_valid_feats.copy()
lr_proba_map_valid = {c: lrprobas_valid[:, j] for j, c in enumerate(lr_classes)}
for iadd in range(NUM_CLUSTS):
    base = lr_proba_map_valid.get(iadd, np.zeros(len(X_valid_base), dtype=np.float64))
    Xy_valid_wLRfeats["LRp" + str(iadd)] = base + lrblur * (
        np.random.rand(len(base)) - 0.5
    )



In [25]:
X = Xy_train_wLRfeats.drop(
    columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore"
).copy()
X.columns = X.columns.astype(str)

LR_FEATURE_COLS = list(X.columns)

# Change (score improvement): use the HBA_probs carried inside Xy_train_feats to guarantee
# perfect row alignment between X and Y (no reset_index drift).
Yprob_train = Xy_train_wLRfeats[HBA_probs].to_numpy(dtype=np.float64)
Yprob_train = np.clip(Yprob_train, 1e-8, 1.0)
Yprob_train = Yprob_train / Yprob_train.sum(axis=1, keepdims=True)

ave_oob = []
nfits = 5

rfmodels = []
for ifit in range(nfits):
    rfmodel = RandomForestRegressor(
        n_estimators=150,
        max_leaf_nodes=int(6 * NUM_CLUSTS),
        max_features=0.5,
        max_samples=0.7,
        oob_score=True,
        n_jobs=-1,
        random_state=GLOBAL_SEED + ifit,
        verbose=0,
    ).fit(X.reindex(columns=LR_FEATURE_COLS), Yprob_train)
    rfmodels.append(rfmodel)
    ave_oob.append(float(rfmodel.oob_score_))

print(
    "RF regressor ave OOB R^2 = {:.3f} +/- {:.3f}".format(
        np.mean(ave_oob), np.std(ave_oob)
    )
)




RF regressor ave OOB R^2 = 0.315 +/- 0.002


In [26]:
def _rf_reg_predict_probs(model, Xdf, eps=1e-8):
    P = model.predict(Xdf).astype(np.float64, copy=False)
    if P.ndim == 1:
        P = P.reshape(-1, HBA_number)
    P = np.nan_to_num(
        P, nan=1.0 / HBA_number, posinf=1.0 / HBA_number, neginf=1.0 / HBA_number
    )
    P = np.clip(P, eps, 1.0)
    P = P / P.sum(axis=1, keepdims=True)
    return P


def _rf_reg_predict_probs_ensemble(models, Xdf, eps=1e-8):
    Ps = []
    for m in models:
        Ps.append(_rf_reg_predict_probs(m, Xdf, eps=eps))
    P = np.mean(np.stack(Ps, axis=0), axis=0)
    P = np.clip(P, eps, 1.0)
    P = P / P.sum(axis=1, keepdims=True)
    return P


X_train_reg = (
    Xy_train_wLRfeats.drop(columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore")
    .copy()
    .reindex(columns=LR_FEATURE_COLS)
)
P_train = _rf_reg_predict_probs_ensemble(rfmodels, X_train_reg)

sol_train = Xy_train_wLRfeats[HBA_probs].copy()
sub_train = pd.DataFrame(P_train, columns=HBA_probs)
print(
    "Train KL (direct 6-prob RF reg, ens): {:.4f}".format(
        kld_score(sol_train, sub_train)
    )
)

X_valid_reg = (
    Xy_valid_wLRfeats.drop(columns=["clust_id", "eeg_id"] + HBA_probs, errors="ignore")
    .copy()
    .reindex(columns=LR_FEATURE_COLS)
)
P_valid = _rf_reg_predict_probs_ensemble(rfmodels, X_valid_reg)
sol_valid = Xy_valid_wLRfeats[HBA_probs].copy()
sub_valid = pd.DataFrame(P_valid, columns=HBA_probs)
print(
    "Valid KL (direct 6-prob RF reg, ens): {:.4f}".format(
        kld_score(sol_valid, sub_valid)
    )
)




Train KL (direct 6-prob RF reg, ens): 0.4925
Valid KL (direct 6-prob RF reg, ens): 1.1996


In [27]:
def find_best_tamed_kl(solution, submission, pred_ids, step=0.02, eps=1e-8):
    """
    Adjust the taming fraction for each cluster center to optimize KL.

    Kept for compatibility with your existing pipeline, but not used for the
    final predictions after switching to direct 6-prob regression.
    """
    mean_all_probs = solution[HBA_votes].to_numpy(dtype=np.float64).mean(axis=0)
    mean_all_probs = np.clip(mean_all_probs, eps, 1.0)
    mean_all_probs = mean_all_probs / mean_all_probs.sum()

    tamed_fracs = 0.0 * np.ones(NUM_CLUSTS, dtype=np.float64)
    tamed_centers = clust_centers.copy()
    for iclust in range(NUM_CLUSTS):
        tamed_centers[iclust, :] = mean_all_probs

    best_fracs = tamed_fracs.copy()
    best_centers = tamed_centers.copy()
    best_kl_global = np.inf

    sol_probs = solution[HBA_votes].to_numpy(dtype=np.float64)
    sol_probs = np.clip(sol_probs, eps, 1.0)
    sol_probs = sol_probs / sol_probs.sum(axis=1, keepdims=True)

    for iclust in range(NUM_CLUSTS):
        last_kl = np.inf
        for this_frac in np.arange(0.0, 1.0 + 1e-12, step):
            tamed_fracs[iclust] = this_frac
            this_cent = (
                tamed_fracs[iclust] * clust_centers[iclust, :]
                + (1.0 - tamed_fracs[iclust]) * mean_all_probs
            )
            this_cent = np.clip(this_cent, eps, 1.0)
            this_cent = this_cent / this_cent.sum()
            tamed_centers[iclust, :] = this_cent

            sub_probs = tamed_centers[pred_ids, :]
            sub_probs = np.clip(sub_probs, eps, 1.0)
            sub_probs = sub_probs / sub_probs.sum(axis=1, keepdims=True)

            this_kl = float(
                np.mean(
                    np.sum(sol_probs * (np.log(sol_probs) - np.log(sub_probs)), axis=1)
                )
            )

            if this_kl < last_kl:
                last_kl = this_kl
                best_fracs = tamed_fracs.copy()
                best_centers = tamed_centers.copy()
                best_kl_global = min(best_kl_global, this_kl)
            else:
                tamed_fracs[iclust] = best_fracs[iclust]
                tamed_centers[iclust, :] = best_centers[iclust, :]
                break

    best_centers = np.clip(best_centers, eps, 1.0)
    best_centers = best_centers / best_centers.sum(axis=1, keepdims=True)
    return best_fracs, best_centers, best_kl_global




In [28]:
def make_submission_valid(df, sample_sub, vote_cols, eps=1e-8):
    """
    Enforce strict Kaggle requirements: correct columns, finiteness, positivity,
    exact row order matching sample_submission, and row sums exactly 1.
    """
    out = df.copy()
    out["eeg_id"] = out["eeg_id"].astype(np.int64)
    for c in vote_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").astype(np.float64)

    order = sample_sub["eeg_id"].astype(np.int64).to_numpy()
    out = out.set_index("eeg_id").reindex(order).reset_index()

    probs = out[vote_cols].to_numpy(dtype=np.float64)
    probs = np.nan_to_num(
        probs,
        nan=1.0 / len(vote_cols),
        posinf=1.0 / len(vote_cols),
        neginf=1.0 / len(vote_cols),
    )

    probs = np.clip(probs, eps, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)

    out[vote_cols] = probs

    assert list(out.columns) == ["eeg_id"] + vote_cols
    assert len(out) == len(sample_sub)
    rs = out[vote_cols].to_numpy(dtype=np.float64).sum(axis=1)
    assert np.all(np.isfinite(rs))
    assert float(np.max(np.abs(rs - 1.0))) < 1e-9
    assert out.isna().sum().sum() == 0
    return out




In [29]:
train_prior = train_meta[HBA_probs].to_numpy(dtype=np.float64)
train_prior = np.clip(train_prior, 1e-8, 1.0)
train_prior = train_prior / train_prior.sum(axis=1, keepdims=True)
EMPIRICAL_PRIOR = train_prior.mean(axis=0)
EMPIRICAL_PRIOR = np.clip(EMPIRICAL_PRIOR, 1e-8, 1.0)
EMPIRICAL_PRIOR = EMPIRICAL_PRIOR / EMPIRICAL_PRIOR.sum()

PRIOR_SMOOTH_ALPHA = 0.08  # was 0.02


def smooth_toward_prior(P, prior, alpha=0.02, eps=1e-8):
    P = np.asarray(P, dtype=np.float64)
    prior = np.asarray(prior, dtype=np.float64).reshape(1, -1)
    Q = (1.0 - alpha) * P + alpha * prior
    Q = np.clip(Q, eps, 1.0)
    Q = Q / Q.sum(axis=1, keepdims=True)
    return Q




In [30]:
sample_sub = pd.read_csv(above_dir + "sample_submission.csv")

Xy_test_feats = assemble_features(
    test_meta, traintest="test", smooth_width=SMOOTH_WIDTH, SHOW_PLOT=False
)

if "eeg_id" in Xy_test_feats.columns:
    assert np.array_equal(
        Xy_test_feats["eeg_id"].astype(np.int64).to_numpy(),
        test_meta["eeg_id"].astype(np.int64).to_numpy(),
    )

X_test_base = Xy_test_feats.drop(columns=["eeg_id"] + HBA_probs, errors="ignore").copy()
X_test_base.columns = X_test_base.columns.astype(str)
X_test_base = X_test_base.reindex(columns=BASE_FEATURE_COLS)

lrprobas = lrmodel.predict_proba(X_test_base)

Xy_test_wLRfeats = Xy_test_feats.copy()
lr_proba_map_test = {c: lrprobas[:, j] for j, c in enumerate(lr_classes)}
for iadd in range(NUM_CLUSTS):
    base = lr_proba_map_test.get(iadd, np.zeros(len(X_test_base), dtype=np.float64))
    Xy_test_wLRfeats["LRp" + str(iadd)] = base + lrblur * (
        np.random.rand(len(base)) - 0.5
    )

X_test_reg = (
    Xy_test_wLRfeats.drop(columns=["eeg_id"] + HBA_probs, errors="ignore")
    .copy()
    .reindex(columns=LR_FEATURE_COLS)
)
test_probs = _rf_reg_predict_probs_ensemble(rfmodels, X_test_reg, eps=1e-8)

test_probs = smooth_toward_prior(
    test_probs, EMPIRICAL_PRIOR, alpha=PRIOR_SMOOTH_ALPHA, eps=1e-8
)

test_submit = pd.DataFrame({"eeg_id": test_meta["eeg_id"].astype(np.int64).to_numpy()})
for j, col in enumerate(HBA_votes):
    test_submit[col] = test_probs[:, j]

test_submit = test_submit[["eeg_id"] + HBA_votes]
test_submit = make_submission_valid(test_submit, sample_sub, HBA_votes, eps=1e-8)

print(test_submit.head())
print("submission shape:", test_submit.shape)

test_submit.to_csv(
    "submission.csv", header=True, index=False, na_rep="", float_format="%.6f"
)
print("Wrote submission.csv")

... 200 done...
... 400 done...
... 600 done...
... 800 done...
... 1000 done...
... 1200 done...
... 1400 done...
... 1600 done...
... 1800 done...
... 2000 done...
... 2200 done...
... 2400 done...
... 2600 done...
... 2800 done...
... 3000 done...
... 3200 done...
... 3400 done...
... 3600 done...
... 3800 done...
... 4000 done...
... 4200 done...
... 4400 done...
... 4600 done...
... 4800 done...
... 5000 done...
... 5200 done...
... 5400 done...
... 5600 done...
... 5800 done...
... 6000 done...
... 6200 done...
... 6400 done...
... 6600 done...
... 6800 done...
... 7000 done...
... 7200 done...
... 7400 done...
... 7600 done...
... 7800 done...
... 8000 done...
... 8200 done...
... 8400 done...
... 8600 done...
... 8800 done...
... 9000 done...
... 9200 done...
... 9400 done...
... 9600 done...
... 9800 done...
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.215069  0.212095  0.090145   0.077836   0.126170   
1  2578018731      0.2150